In [2]:
!ls -lh ~/esai_2026/data/NHAMCS/

total 43M
-rwxr-x--- 1 kw468 esai_2026  3.6M Sep  2 15:01 desc.txt
-rwxr-x--- 1 kw468 esai_2026   38M Sep  2 15:01 nhamcsed2015.csv
-rwxr-x--- 1 kw468 esai_2026 1014K Sep  2 15:01 nhamcsed2015.pdf


In [3]:
import pandas as pd

df = pd.read_csv("~/esai_2026/data/NHAMCS/nhamcsed2015.csv")
print(f"Shape: {df.shape}")

Shape: (21061, 1031)


/tmp/ipykernel_3047716/1197564808.py:3: DtypeWarning: Columns (59,60,75,76,270,271,272,273,274,275,276,277,278,279,585,605,625,645,665,685,705,725,745,765,785,805,825,845,865,885,905,925,945,965,985,1005) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("~/esai_2026/data/NHAMCS/nhamcsed2015.csv")


In [5]:
missing = df.isna().sum().sort_values(ascending=False)
print(missing.head(20))  # worst overall

key_cols = ['AGE','SEX','ETHIM','WAITTIME','LOV','PATWT','VDAYR','ARRTIME',
            'PAYTYPER','HTN','DIABTYP2','OBESITY','DEPRN','TOTCHRON',
            'INJPOISAD','INTENT15','ANYIMAGE','NUMMED']
print(df[key_cols].isna().sum())

RX29V2C4    21061
RX30V3C4    21061
RX30V1C3    21061
RX30CAT3    21061
RX30V1C4    21061
RX30V2C4    21061
RX30V2C3    21061
RX29V3C3    21061
RX29V3C2    21061
RX29V2C3    21061
RX30CAT4    21061
RX27V1C3    21061
RX27CAT3    21061
RX27CAT2    21061
RX28V2C3    21061
RX17V3C3    21061
RX29V3C4    21061
RX17V3C4    21061
RX29V1C3    21061
RX25V3C4    21061
dtype: int64
AGE          0
SEX          0
ETHIM        0
WAITTIME     0
LOV          0
PATWT        0
VDAYR        0
ARRTIME      0
PAYTYPER     0
HTN          0
DIABTYP2     0
OBESITY      0
DEPRN        0
TOTCHRON     0
INJPOISAD    0
INTENT15     0
ANYIMAGE     0
NUMMED       0
dtype: int64


In [6]:
for col in ['AGE','WAITTIME','LOV','PATWT','TOTCHRON','NUMMED']:
    print(col)
    print(df[col].value_counts().sort_index().head(5))
    print(df[col].value_counts().sort_index().tail(5))
    print()

AGE
AGE
0    586
1    483
2    345
3    277
4    245
Name: count, dtype: int64
AGE
89     69
90     65
91     47
92     48
93    155
Name: count, dtype: int64

WAITTIME
WAITTIME
-9    3196
-7     712
 0    1201
 1     561
 2     420
Name: count, dtype: int64
WAITTIME
1105    1
1114    1
1140    1
1224    1
1305    1
Name: count, dtype: int64

LOV
LOV
-9    1480
 0      19
 1       6
 2       2
 3       2
Name: count, dtype: int64
LOV
5560    1
5567    1
5672    1
5687    1
5732    1
Name: count, dtype: int64

PATWT
PATWT
117    23
157    34
173    20
184     3
231    31
Name: count, dtype: int64
PATWT
30377    57
33413    16
35714    51
39245    30
42002    26
Name: count, dtype: int64

TOTCHRON
TOTCHRON
-9      339
 0    10662
 1     4474
 2     2415
 3     1496
Name: count, dtype: int64
TOTCHRON
7     128
8      48
9      27
10     16
12      1
Name: count, dtype: int64

NUMMED
NUMMED
0    4281
1    4731
2    4088
3    2869
4    1897
Name: count, dtype: int64
NUMMED
26    7
27    7
2

In [7]:
n_unweighted = len(df)
print(f"Unweighted visits (sample size): {n_unweighted:,}")

weighted_total = df['PATWT'].sum()
print(f"Weighted total estimated U.S. ED visits (2015): {weighted_total:,.0f}")

Unweighted visits (sample size): 21,061
Weighted total estimated U.S. ED visits (2015): 136,943,181


In [8]:
df['WAITTIME_clean'] = df['WAITTIME'].replace({-9: pd.NA, -7: pd.NA})
df['LOV_clean']       = df['LOV'].replace({-9: pd.NA})

In [9]:
import numpy as np

def weighted_mean(x, w):
    x = x.dropna()
    w = w.loc[x.index]
    return np.average(x, weights=w)

def weighted_median(x, w):
    x = x.dropna()
    w = w.loc[x.index]
    order = np.argsort(x.values)
    x_sorted = x.values[order]
    w_sorted = w.values[order]
    cum_w = np.cumsum(w_sorted)
    cutoff = cum_w[-1] / 2.0
    idx = np.searchsorted(cum_w, cutoff)
    return x_sorted[idx]

In [10]:
# (a) Age
print("AGE")
print(f"  Unweighted — mean: {df['AGE'].mean():.2f}, median: {df['AGE'].median():.1f}")
print(f"  Weighted   — mean: {weighted_mean(df['AGE'], df['PATWT']):.2f}, median: {weighted_median(df['AGE'], df['PATWT']):.1f}")

# (b) Sex and ethnicity — unweighted vs weighted shares
print("\nSEX — unweighted share:")
print(df['SEX'].value_counts(normalize=True))
print("SEX — weighted share:")
print(df.groupby('SEX')['PATWT'].sum() / df['PATWT'].sum())

print("\nETHIM — unweighted share:")
print(df['ETHIM'].value_counts(normalize=True))
print("ETHIM — weighted share:")
print(df.groupby('ETHIM')['PATWT'].sum() / df['PATWT'].sum())

# (c) Median wait time (cleaned)
print("\nWAITTIME (cleaned)")
print(f"  Unweighted median: {df['WAITTIME_clean'].median():.1f}")
print(f"  Weighted median:   {weighted_median(df['WAITTIME_clean'], df['PATWT']):.1f}")

# (d) Average length of visit (cleaned)
print("\nLOV (cleaned)")
print(f"  Unweighted mean: {df['LOV_clean'].mean():.1f}")
print(f"  Weighted mean:   {weighted_mean(df['LOV_clean'], df['PATWT']):.1f}")

AGE
  Unweighted — mean: 37.56, median: 34.0
  Weighted   — mean: 37.04, median: 34.0

SEX — unweighted share:
SEX
1    0.551256
2    0.448744
Name: proportion, dtype: float64
SEX — weighted share:
SEX
1    0.554364
2    0.445636
Name: PATWT, dtype: float64

ETHIM — unweighted share:
ETHIM
2    0.841223
1    0.158777
Name: proportion, dtype: float64
ETHIM — weighted share:
ETHIM
1    0.164934
2    0.835066
Name: PATWT, dtype: float64

WAITTIME (cleaned)
  Unweighted median: 19.0
  Weighted median:   18.0

LOV (cleaned)
  Unweighted mean: 221.9
  Weighted mean:   213.7


In [12]:
df['arr_hour'] = df['ARRTIME'].astype(str).str.zfill(4).str[:2].astype(int)

day_labels = {1:'Sunday', 2:'Monday', 3:'Tuesday', 4:'Wednesday',
              5:'Thursday', 6:'Friday', 7:'Saturday'}

by_day = df.groupby('VDAYR')['PATWT'].sum().rename(index=day_labels)
print(by_day.sort_values(ascending=False))

by_hour = df.groupby('arr_hour')['PATWT'].sum()
print(by_hour.sort_index())

print("Busiest day:", by_day.idxmax(), "-", f"{by_day.max():,.0f}")
print("Busiest hour:", by_hour.idxmax(), "-", f"{by_hour.max():,.0f}")

VDAYR
Monday       22452845
Tuesday      20568287
Wednesday    19723821
Friday       18784209
Thursday     18743470
Saturday     18672207
Sunday       17998342
Name: PATWT, dtype: int64
arr_hour
0     4506626
1     2473330
2     2198571
3     1917072
4     1835272
5     1820212
6     2377504
7     3576795
8     4862647
9     6848652
10    8193821
11    7792006
12    8238141
13    7666168
14    7859572
15    7597133
16    7655578
17    7809440
18    8589298
19    7995955
20    7394421
21    6766962
22    6286263
23    4681742
Name: PATWT, dtype: int64
Busiest day: Monday - 22,452,845
Busiest hour: 18 - 8,589,298


In [13]:
pay_labels = {-9: 'Blank/Missing', -8: 'Unknown', 1: 'Private insurance', 2: 'Medicare',
              3: 'Medicaid/CHIP', 4: "Worker's comp", 5: 'Self-pay', 6: 'No charge/Charity',
              7: 'Other'}

weighted_pay = (df.groupby('PAYTYPER')['PATWT'].sum() / df['PATWT'].sum()) \
                 .rename(index=pay_labels).sort_values(ascending=False)
print(weighted_pay)

for label in ['Private insurance', 'Medicare', 'Medicaid/CHIP', 'Self-pay']:
    print(f"{label}: {weighted_pay[label]:.1%}")

PAYTYPER
Medicaid/CHIP        0.311517
Private insurance    0.275729
Medicare             0.177415
Self-pay             0.090330
Unknown              0.082959
Blank/Missing        0.024685
Other                0.022055
No charge/Charity    0.008321
Worker's comp        0.006988
Name: PATWT, dtype: float64
Private insurance: 27.6%
Medicare: 17.7%
Medicaid/CHIP: 31.2%
Self-pay: 9.0%


In [17]:
df['TOTCHRON_clean'] = df['TOTCHRON'].replace({-9: pd.NA})

chronic_cols = ['ETOHAB','ALZHD','ASTHMA','CANCER','CEBVD','CKD','COPD','CHF','CAD','DEPRN',
                'DIABTYP1','DIABTYP2','DIABTYP0','ESRD','HPE','EDHIV','HYPLIPID','HTN','OBESITY',
                'OSA','OSTPRSIS','SUBSTAB']

# (a) Weighted share of visits with at least one chronic condition
has_chronic = (df[chronic_cols] == 1).any(axis=1)
weighted_share_chronic = (has_chronic * df['PATWT']).sum() / df['PATWT'].sum()
print(f"Weighted share with >=1 chronic condition: {weighted_share_chronic:.1%}")

# Cross-check against TOTCHRON_clean > 0 (excluding the 339 "blank section" rows)
mask = df['TOTCHRON_clean'].notna()
cross_check = ((df.loc[mask, 'TOTCHRON_clean'] > 0) * df.loc[mask, 'PATWT']).sum() / df.loc[mask, 'PATWT'].sum()
print(f"Cross-check via TOTCHRON_clean > 0: {cross_check:.1%}")

# (b) Most prevalent individual chronic condition (weighted)
prevalence = {}
for col in chronic_cols:
    prevalence[col] = (df[col] == 1).mul(df['PATWT']).sum() / df['PATWT'].sum()
prevalence = pd.Series(prevalence).sort_values(ascending=False)
print(prevalence.head(10))

Weighted share with >=1 chronic condition: 47.0%
Cross-check via TOTCHRON_clean > 0: 47.6%
HTN         0.236378
ASTHMA      0.098470
DEPRN       0.093245
HYPLIPID    0.080558
SUBSTAB     0.065601
CAD         0.059998
DIABTYP0    0.056878
COPD        0.052947
DIABTYP2    0.046332
OBESITY     0.035898
dtype: float64


In [18]:
injpois_labels = {-9: 'Blank', -8: 'Unknown', 1: 'Injury/trauma', 2: 'Overdose/poisoning',
                   3: 'Adverse effect of medical/surgical tx', 4: 'Not related'}

weighted_injpois = (df.groupby('INJPOISAD')['PATWT'].sum() / df['PATWT'].sum()) \
                     .rename(index=injpois_labels).sort_values(ascending=False)
print(weighted_injpois)

frac_injury_related = weighted_injpois[['Injury/trauma', 'Overdose/poisoning',
                                          'Adverse effect of medical/surgical tx']].sum()
print(f"\n(a) Weighted share related to injury/overdose/adverse effect: {frac_injury_related:.1%}")

mask_injury = df['INJPOISAD'].isin([1, 2, 3])
intent_labels = {-9: 'Blank', -8: 'Unknown/unclear', 1: 'Intentional',
                  2: 'Unintentional', 4: 'Questionable'}
weighted_intent = (df.loc[mask_injury].groupby('INTENT15')['PATWT'].sum()
                   / df.loc[mask_injury, 'PATWT'].sum()) \
                     .rename(index=intent_labels).sort_values(ascending=False)
print("\n(b) Among injury/overdose/adverse-effect visits:")
print(weighted_intent)

INJPOISAD
Not related                              0.626705
Injury/trauma                            0.294868
Unknown                                  0.037351
Adverse effect of medical/surgical tx    0.022205
Overdose/poisoning                       0.013765
5                                        0.002894
Blank                                    0.002212
Name: PATWT, dtype: float64

(a) Weighted share related to injury/overdose/adverse effect: 33.1%

(b) Among injury/overdose/adverse-effect visits:
INTENT15
Unintentional      0.684038
Blank              0.257377
Intentional        0.053727
Unknown/unclear    0.004857
Name: PATWT, dtype: float64


In [19]:
injpois_labels = {-9: 'Blank', -8: 'Unknown', 1: 'Injury/trauma', 2: 'Overdose/poisoning',
                   3: 'Adverse effect of medical/surgical tx', 4: 'Not related', 5: 'Questionable'}
crosstab = pd.crosstab(df['INJPOISAD'].map(injpois_labels), df['INTENT15'], normalize='index')
print(crosstab)

INTENT15                                     -9        -8         1         2  \
INJPOISAD                                                                       
Adverse effect of medical/surgical tx  1.000000  0.000000  0.000000  0.000000   
Blank                                  1.000000  0.000000  0.000000  0.000000   
Injury/trauma                          0.220220  0.004624  0.056131  0.719024   
Not related                            1.000000  0.000000  0.000000  0.000000   
Overdose/poisoning                     0.017921  0.000000  0.121864  0.860215   
Questionable                           0.000000  0.000000  0.000000  0.000000   
Unknown                                1.000000  0.000000  0.000000  0.000000   

INTENT15                                 4  
INJPOISAD                                   
Adverse effect of medical/surgical tx  0.0  
Blank                                  0.0  
Injury/trauma                          0.0  
Not related                            0.0  


In [20]:
mask_intent_applicable = df['INJPOISAD'].isin([1, 2])  # Injury/trauma, Overdose/poisoning only
intent_labels = {-9: 'Blank', -8: 'Unknown/unclear', 1: 'Intentional', 2: 'Unintentional', 4: 'Questionable'}
weighted_intent_clean = (df.loc[mask_intent_applicable].groupby('INTENT15')['PATWT'].sum()
                         / df.loc[mask_intent_applicable, 'PATWT'].sum()) \
                          .rename(index=intent_labels).sort_values(ascending=False)
print(weighted_intent_clean)

INTENT15
Unintentional      0.733251
Blank              0.203949
Intentional        0.057593
Unknown/unclear    0.005207
Name: PATWT, dtype: float64


In [21]:
# (a) Any imaging
weighted_anyimage = (df['ANYIMAGE'] == 1).mul(df['PATWT']).sum() / df['PATWT'].sum()
print(f"(a) Weighted share with any imaging: {weighted_anyimage:.1%}")

# (b) Among visits with blood tests, which were most frequently ordered
blood_test_cols = ['ABG','BAC','BMP','BLOODCX','BNP','BUNCREAT','CARDENZ','CBC','CMP','DDIMER',
                    'ELECTROL','GLUCOSE','LACTATE','LFT','PTTINR','OTHERBLD']

has_blood_test = (df[blood_test_cols] == 1).any(axis=1)
weighted_share_blood = (has_blood_test * df['PATWT']).sum() / df['PATWT'].sum()
print(f"Weighted share of visits with >=1 blood test: {weighted_share_blood:.1%}")

blood_test_freq = {}
denom = (has_blood_test * df['PATWT']).sum()
for col in blood_test_cols:
    blood_test_freq[col] = ((df[col] == 1) & has_blood_test).mul(df['PATWT']).sum() / denom
blood_test_freq = pd.Series(blood_test_freq).sort_values(ascending=False)
print(blood_test_freq)

(a) Weighted share with any imaging: 47.0%
Weighted share of visits with >=1 blood test: 42.4%
CBC         0.854264
CMP         0.553118
OTHERBLD    0.461064
BMP         0.251837
GLUCOSE     0.191559
PTTINR      0.177609
BUNCREAT    0.157205
CARDENZ     0.097876
LFT         0.097137
ELECTROL    0.076591
BLOODCX     0.069980
BNP         0.059311
DDIMER      0.055659
BAC         0.040440
ABG         0.037456
LACTATE     0.030833
dtype: float64


In [22]:
# (a) Mean and median number of medications per visit (weighted)
mean_nummed = weighted_mean(df['NUMMED'], df['PATWT'])
median_nummed = weighted_median(df['NUMMED'], df['PATWT'])
print(f"Weighted mean NUMMED: {mean_nummed:.2f}")
print(f"Weighted median NUMMED: {median_nummed:.1f}")
print(f"(for comparison) Unweighted mean/median: {df['NUMMED'].mean():.2f} / {df['NUMMED'].median():.1f}")

# (b) Share of visits with 0, 1-2, and 3+ medications (weighted)
def med_bucket(n):
    if n == 0:
        return '0 medications'
    elif n <= 2:
        return '1-2 medications'
    else:
        return '3+ medications'

df['med_bucket'] = df['NUMMED'].apply(med_bucket)
weighted_bucket = df.groupby('med_bucket')['PATWT'].sum() / df['PATWT'].sum()
print(weighted_bucket.sort_values(ascending=False))

Weighted mean NUMMED: 2.49
Weighted median NUMMED: 2.0
(for comparison) Unweighted mean/median: 2.54 / 2.0
med_bucket
1-2 medications    0.419073
3+ medications     0.371500
0 medications      0.209427
Name: PATWT, dtype: float64


In [24]:
import numpy as np

df['WAITTIME_clean']  = df['WAITTIME'].replace({-9: np.nan, -7: np.nan})
df['LOV_clean']       = df['LOV'].replace({-9: np.nan})
df['TOTCHRON_clean']  = df['TOTCHRON'].replace({-9: np.nan})

blood_test_cols = ['ABG','BAC','BMP','BLOODCX','BNP','BUNCREAT','CARDENZ','CBC','CMP','DDIMER',
                    'ELECTROL','GLUCOSE','LACTATE','LFT','PTTINR','OTHERBLD']
df['has_blood_test'] = (df[blood_test_cols] == 1).any(axis=1).astype(int)
df['died_ed'] = (df['HDSTAT'] == 2).astype(int)

corr_vars = ['AGE', 'TOTCHRON_clean', 'NUMMED', 'WAITTIME_clean', 'LOV_clean', 'ANYIMAGE',
             'has_blood_test', 'DOA', 'DIEDED', 'died_ed']

corr_matrix = df[corr_vars].corr()
print(corr_matrix.round(2))

                 AGE  TOTCHRON_clean  NUMMED  WAITTIME_clean  LOV_clean  \
AGE             1.00            0.54    0.19           -0.01       0.13   
TOTCHRON_clean  0.54            1.00    0.23           -0.01       0.15   
NUMMED          0.19            0.23    1.00            0.01       0.19   
WAITTIME_clean -0.01           -0.01    0.01            1.00       0.23   
LOV_clean       0.13            0.15    0.19            0.23       1.00   
ANYIMAGE        0.25            0.19    0.19           -0.01       0.12   
has_blood_test  0.34            0.32    0.29            0.02       0.26   
DOA             0.01           -0.00   -0.01           -0.00      -0.01   
DIEDED          0.03            0.01    0.01           -0.01      -0.00   
died_ed         0.06            0.05    0.05           -0.02       0.01   

                ANYIMAGE  has_blood_test   DOA  DIEDED  died_ed  
AGE                 0.25            0.34  0.01    0.03     0.06  
TOTCHRON_clean      0.19            0.32 -